# 5.4 Parallel Execution & Map-Reduce

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

**Prerequisites:** 4.2 (reducers, the replace-by-default rule), 5.3 (`Command` and `Send` from `langgraph.types`, subgraph composition). All demos run with no API key.

## Learning objectives

- Explain the super-step guarantee: independent nodes run concurrently, and one checkpoint covers the whole step.
- Fan out dynamically with `Send` from a conditional edge, and state exactly what the target node receives as its input.
- Build a map-reduce pipeline: fan out over N documents, merge with a reducer, reduce to a final answer.
- Predict what happens when one parallel branch fails, and apply the two documented remedies.
- Decide when **not** to parallelize, and separate real throughput gains from latency-perception tricks.

## What "parallel" means in LangGraph

Nodes with no dependency between them run in the same **super-step**: one tick of the graph in which every scheduled node executes, followed by a single checkpoint covering all of their updates. Two facts make this concrete, and both were verified by running code on the pinned versions.

First, the execution is genuinely concurrent. Two nodes that each sleep 0.4 seconds finish in about 0.42 seconds wall-clock, not 0.8:

In [ ]:
import operator
import time
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class P(TypedDict):
    log: Annotated[list, operator.add]

def slow_b(state: P) -> dict:
    time.sleep(0.4)
    return {"log": ["b"]}

def slow_c(state: P) -> dict:
    time.sleep(0.4)
    return {"log": ["c"]}

builder = StateGraph(P)
builder.add_node("b", slow_b)
builder.add_node("c", slow_c)
builder.add_edge(START, "b")
builder.add_edge(START, "c")
builder.add_edge("b", END)
builder.add_edge("c", END)
graph = builder.compile()

t0 = time.time()
out = graph.invoke({"log": []})
print(round(time.time() - t0, 2), "seconds |", out["log"])

Expected output:

In [ ]:
0.42 seconds | ['b', 'c']

Synchronous nodes run on a thread pool, so CPU-free waiting (network calls, model calls, sleeps) overlaps. CPU-bound Python work still shares the GIL, which matters when your "parallel" nodes are doing heavy local computation rather than waiting.

Second, one checkpoint covers the step. Streaming the same shape of graph with `stream_mode="updates"` yields one chunk per node completion (`{'b': ...}`, then `{'c': ...}`), but the checkpoint history shows a single checkpoint containing both updates. The stream tells you when each node finished; the checkpoint tells you the step was atomic. Keep those two views separate in your head: completion order in the stream is timing-dependent, while the checkpoint is all-or-nothing.

## The stream order versus merge order distinction

This is the subtlety that bites in production, so it gets its own section. Run two async nodes where `b` sleeps 0.01s and `a` sleeps 0.05s, and stream with `updates` mode: the chunks arrive `b` then `a`, every run, because `b` genuinely finishes first. The stream reflects completion timing. But the merged state is **not** timing-dependent. With a reducer attached, the merged list order follows the task scheduling order, and it is stable across runs: twenty runs with randomised sleep jitter still merge Send fan-out results in fan-out order (`doc1` through `doc8`), never in completion order.

The practical rule: never read ordering guarantees out of a stream, and never assume the merged state preserves arrival order. If the final order matters to the user (a ranked list, a timeline), sort explicitly in the reduce step.

## `Send`: dynamic fan-out

Fixed edges parallelise a known set of branches. `Send` parallelises a set you only learn at runtime: one task per document, one per search query, one per customer. It is imported from `langgraph.types` (verified on the pinned versions alongside `Command` in lesson 4.2's notes):

In [ ]:
from langgraph.types import Send

The pattern is a conditional edge whose router returns a list of `Send` objects instead of a node name. The full working version is the map-reduce example in the next section; its core is two lines:

- the router: `return [Send("summarize", {"doc_id": d["id"], "text": d["text"]}) for d in state["docs"]]`
- the wiring: `builder.add_conditional_edges(START, fan_out, ["summarize"])`

Each `Send("summarize", payload)` schedules one execution of the `summarize` node with `payload` as its input, and all of them run in the same super-step. The list in `add_conditional_edges` declares the possible targets, exactly like `destinations=` did for `Command` in lesson 5.3.

The critical, verified fact: **the Send payload is the node's entire input.** The target node does not see the parent state. If the payload is `{"doc_id": ..., "text": ...}`, then inside the node `state` contains exactly those two keys, no `docs`, no sibling fields. This was verified by inspecting the keys each task received. When a worker needs parent context, merge it into the payload yourself with `Send("worker", {**state, "task": task_description})`.

Forgetting this is the most common `Send` bug: the node reads `state["docs"]` and gets a `KeyError`, because the payload replaced the state rather than extending it.

## Map-reduce over documents

Map-reduce is the canonical `Send` application. Map: one task per document, each producing a small result. Reduce: one node that reads all the mapped results and synthesises the answer. The reducer on the results field is what makes the parallel writes legal (lesson 4.2's rule: replacement is the default; parallel branches writing one plain key is an error).

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class DocState(TypedDict):
    docs: list[dict]
    summaries: Annotated[list, operator.add]   # the reducer: load-bearing
    report: str

def fan_out(state: DocState):
    return [Send("summarize", {"doc_id": d["id"], "text": d["text"]})
            for d in state["docs"]]

def summarize(state: dict) -> dict:
    words = state["text"].split()
    return {"summaries": [f"{state['doc_id']}: {len(words)} words, opens with '{words[0]}'"]}

def reduce(state: DocState) -> dict:
    lines = "\n".join(f"- {s}" for s in sorted(state["summaries"]))
    return {"report": f"Digest of {len(state['summaries'])} documents:\n{lines}"}

builder = StateGraph(DocState)
builder.add_node("summarize", summarize)
builder.add_node("reduce", reduce)
builder.add_conditional_edges(START, fan_out, ["summarize"])
builder.add_edge("summarize", "reduce")
builder.add_edge("reduce", END)
graph = builder.compile()

docs = [
    {"id": "doc1", "text": "LangGraph graphs are state machines with checkpoints"},
    {"id": "doc2", "text": "Send enables dynamic fan out at runtime for map reduce"},
    {"id": "doc3", "text": "Reducers merge parallel writes safely into one channel"},
    {"id": "doc4", "text": "Supervisors route work between specialised workers"},
]
print(graph.invoke({"docs": docs, "summaries": [], "report": ""})["report"])

Expected output:

In [ ]:
Digest of 4 documents:
- doc1: 7 words, opens with 'LangGraph'
- doc2: 10 words, opens with 'Send'
- doc3: 8 words, opens with 'Reducers'
- doc4: 6 words, opens with 'Supervisors'

Read the shape of the run. `fan_out` runs once and produces four `Send` packets. Four `summarize` tasks run in one super-step, each returning one string into the `summaries` reducer. Only when all four complete does `reduce` run, reading the merged list. The edge `summarize -> reduce` is the barrier: the reduce node waits for the whole fan, not for the first finisher. In production the `summarize` node holds the model call (one document per call, small context, cheap), and `reduce` holds the synthesis call over the distilled summaries. That is the whole economic argument for map-reduce with LLMs: N small-context calls in parallel instead of one giant-context call, each one cheaper and less prone to losing the middle of the context.

## Why the reducer is load-bearing

Remove the reducer and point two parallel branches at one plain field, and the framework refuses:

In [ ]:
class NR(TypedDict):
    val: str

def w1(state: NR) -> dict: return {"val": "from-w1"}
def w2(state: NR) -> dict: return {"val": "from-w2"}

builder = StateGraph(NR)
builder.add_node("w1", w1)
builder.add_node("w2", w2)
builder.add_edge(START, "w1")
builder.add_edge(START, "w2")
builder.add_edge("w1", END)
builder.add_edge("w2", END)
# EXPECTED-FAILURE: raises InvalidUpdateError, "Can receive only one value per step"
builder.compile().invoke({"val": "seed"})

This raises:

In [ ]:
langgraph.errors.InvalidUpdateError: At key 'val': Can receive only one value per step.
Use an Annotated key to handle multiple values.

The super-step is transactional: with a plain (last-value) channel the framework cannot decide which write wins, so it errors rather than guessing. This is lesson 4.2's replace-versus-reduce semantics under concurrency, and it is a feature: silent last-writer-wins across parallel branches would be a data-loss bug wearing a successful run. The fix is always the same: declare the merge rule in the schema with `Annotated[..., operator.add]` (or `add_messages` for message lists, or a custom two-argument reducer), and initialise the field in the invoke input so the reducer has something to merge into.

## When one branch fails

The transactional super-step cuts both ways. If one of four `Send` tasks raises, the whole run raises; none of the updates from that step are applied:

In [ ]:
def flaky_work(state: dict) -> dict:
    if state["doc_id"] == "doc2":
        raise ValueError("doc2 is corrupt")
    return {"results": [f"ok:{state['doc_id']}"]}

Invoking the fan-out over four documents raises `ValueError: doc2 is corrupt`, and `results` contains nothing, not even the three successes. The official docs state this plainly: the entire super-step errors, and no partial updates land in state.

There are two remedies, and they compose.

**Catch it in the node.** Ordinary Python around the risky call, returning an error marker into the reducer instead of raising. The reduce step then sees which branches failed and can report, skip, or escalate. This is the right default for expected failures: a corrupt document, a 404 from one source, a parse error in one chunk.

**Retry the failing branch.** Pass a `RetryPolicy` to `add_node`:

In [ ]:
from langgraph.types import RetryPolicy

builder.add_node("work", flaky_work,
                 retry_policy=RetryPolicy(max_attempts=3, retry_on=ValueError))

Two verified details matter here. First, only the failing branch is retried: in a four-document fan-out where `doc2` fails once then succeeds, the attempt counts are `doc1: 1, doc2: 2, doc3: 1, doc4: 1`. The successes are not re-run. Second, the default `retry_on` only covers network errors (`ConnectionError`, HTTP 5xx); a `ValueError` is **not** retried unless you pass `retry_on` explicitly, which was verified by watching the retry not happen and then happen after adding the argument. With a checkpointer attached, successful branches' results are saved and are not repeated on resume either.

Choose by failure type: retry for transient faults (rate limits, timeouts, 5xx), catch-and-mark for permanent per-item faults (corrupt input, 404s, schema violations). Retrying a corrupt document three times is just paying to fail slowly.

## When NOT to parallelize

Parallelism has a cost ledger, and several rows are easy to forget:

- **Sequential dependencies.** If step B needs step A's output, there is nothing to parallelise. Fanning out and then re-joining on data you already had just adds scheduling overhead.
- **Rate limits.** Four parallel model calls hit a rate-limited API four times at once. A 429 on one branch fails the super-step (previous section), so an unthrottled fan-out can turn a working pipeline into a failing one. Cap fan-out width or add backoff when the downstream has limits.
- **Cost multiplies.** Four parallel LLM calls cost four calls. Map-reduce over 100 documents is 100 map calls plus the reduce call. The parallelism buys wall-clock time, not money; budget the fan-out width like any other spend.
- **Shared mutable resources.** Branches that write to the same file, the same database row, or the same external system without coordination race. The state reducer only arbitrates LangGraph channels, not the outside world.
- **Debuggability.** A failing branch inside a 50-wide fan-out is harder to reproduce and inspect than a failing step in a sequence. Keep fan-outs narrow while developing; widen them once the per-item logic is solid.
- **Determinism requirements.** If the downstream consumer needs items in a guaranteed order, you must sort in the reduce step (the merge-order section above). Do not rely on completion order.

## Latency perception versus real throughput

Two honest statements. First, parallelism reduces wall-clock time only when the bottleneck is waiting: network calls, model calls, disk reads. The 0.42-seconds-for-two-0.4-second-nodes demo is pure waiting overlap. If the nodes are CPU-bound Python, the GIL serialises them and the fan-out buys nothing but overhead. Profile before assuming.

Second, a fan-out is bounded by its slowest branch. Four documents where three summarise in 2 seconds and one takes 30 seconds finish in 30 seconds plus the reduce step. Parallelism converts the total from the **sum** of branch times to the **maximum** of branch times, and the maximum is set by the worst item in the batch. When branch times vary wildly, consider whether the slow items need their own handling (timeout, separate queue) rather than holding the whole fan hostage. Streaming (lesson 4.4) fixes how the wait *feels*; parallelism fixes how long the wait *is*; neither fixes a fundamentally slow branch.

## What can go wrong

- **Parallel writes to a non-reducer field.** Two branches returning the same plain key raise `InvalidUpdateError: Can receive only one value per step`. The fix is the reducer in the schema, not in the nodes. This is the single most common map-reduce compile-that-runs-then-fails.
- **Forgetting the reducer field in the invoke input.** The reducer merges into the existing value; if the input omits the field there is nothing to merge into. Initialise every reducer field (`"summaries": []`) at invoke time.
- **Reading parent state inside a `Send` target.** The payload is the whole input; `state["docs"]` inside the worker is a `KeyError`. Merge needed parent context into the payload at the fan-out site.
- **One bad branch kills the batch.** The super-step is transactional: one exception, zero updates applied, the run raises. Catch expected per-item failures in the node and mark them; reserve `RetryPolicy` for transient faults, and remember the default `retry_on` skips non-network errors.
- **Unbounded fan-out against rate limits.** A 200-document fan-out against a rate-limited API fails branches, and failed branches fail the run. Cap the width, or batch the fan-out.
- **Assuming stream order is merge order.** Stream chunks arrive in completion order; the reducer merges in scheduling order. Code that indexes "the first result" from a stream is reading timing, not data. Sort in the reduce step when order matters.
- **Parallelising a disguised sequence.** If the reduce logic secretly needs branch A's output to process branch B, the fan-out is a sequence with extra steps and extra failure modes. Draw the data dependencies before reaching for `Send`.
- **Checkpoint bloat on wide fans.** Every branch's writes persist through the checkpointer. A 500-wide fan-out over large documents writes 500 payloads into checkpoint storage. For the EU audience this is also a data-residency note: fan-out payloads containing personal data are stored per branch, so retention policy must cover the fan, not just the final report.

## Check your understanding

**1. Two independent nodes each sleep 0.4 seconds. The run takes 0.42 seconds, yet `stream_mode="updates"` yields two separate chunks. Reconcile these observations.**
The 0.42-second wall time shows the nodes ran concurrently in one super-step (thread pool), followed by a single checkpoint covering both updates. The two stream chunks reflect completion events, not super-steps: `updates` mode yields one chunk per node as each finishes. Concurrency is a property of the super-step; chunking is a property of the stream.

**2. A `Send` target node does `state["docs"]` and raises `KeyError`. The parent state has `docs`. What went wrong?**
The `Send` payload is the node's entire input; the parent state is not merged. The node received only the payload (e.g. `{"doc_id": ..., "text": ...}`), so `state["docs"]` does not exist inside it. Fix: include the needed parent context in the payload at the fan-out site, e.g. `Send("worker", {**state, "task": ...})`.

**3. Four parallel branches write to a plain `str` field `val`. What happens, and what is the exact fix?**
The run raises `langgraph.errors.InvalidUpdateError: At key 'val': Can receive only one value per step`. The super-step is transactional and the framework refuses to pick a winner. Fix: declare the merge rule in the schema, e.g. `val: Annotated[list, operator.add]`, have each branch return a one-element list, and initialise the field in the invoke input.

**4. In a four-document `Send` fan-out, the `doc2` branch raises `ValueError` on its first attempt and succeeds on retry. You attached `RetryPolicy(max_attempts=3, retry_on=ValueError)` to the worker node. Describe the final state and the attempt counts.**
The run succeeds. Attempt counts are `doc1: 1, doc2: 2, doc3: 1, doc4: 1`: only the failing branch is retried, the successes are not re-run. The `retry_on=ValueError` is required because the default retry predicate only covers network errors; without it the `ValueError` propagates and the whole super-step fails with nothing applied.

**5. A map-reduce over 100 documents runs no faster than the sequential version, and the API bill tripled. Name two likely causes from this lesson's cost ledger.**
One: the bottleneck was not waiting (CPU-bound work under the GIL does not overlap), so the fan-out added scheduling overhead without concurrency. Two: the fan-out hit rate limits, causing branch failures and retries, each retry billed as another call; or simply that 100 parallel model calls cost 100 calls while the sequential version's cost was misremembered. Parallelism buys wall-clock time, never money.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.